In [1]:
import importlib as imp
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import utils2p.synchronization
import yaml
from pathlib import Path
from scipy import signal
from rye2p import thorsync
# from rye2p.thorimage.settings import thorimage_settings
from rye2p.thorsync import linenames
import copy

# set upstairs or downstairs microscope
microscope = 'upstairs-2p'

# set project directory paths
PRJ_DATA_DIR = Path("/local/matrix/Remy-Data/projects/natural_mixtures")
RAW_DATA_DIR = Path("/local/tensor-archive/Remy/natural_mixtures/raw_data")
NAS_PROC_DIR = PRJ_DATA_DIR.joinpath('processed_data')

In [2]:
print(matplotlib.get_configdir())
matplotlib.style.reload_library()
matplotlib.style.use('remy-default')


/home/remy/.config/matplotlib


In [3]:
def load_thorsync_line_mapping(microscope_name):
    """Load ThorSync line name mapping config"""
    line_config_folder = Path("/home/remy/PycharmProjects/rye2p/data/microscopes")

    thorsync_line_file = line_config_folder.joinpath(
            f"{microscope_name}/thorsync_lines.yaml"
            )

    with open(thorsync_line_file, 'r') as f:
        matched_lines = yaml.safe_load(f)

    return matched_lines

# ThorSync pre-processing demo

We convert .raw ThorImage movies into more easily readable .tif stacks, and save them to a processed data folder.

The Experiment.xml file is also copied to the processed data folder, and important information about the ThorImage
recording is saved to `thorimage_settings.yaml`.

# Set dataset info

ThorImage and ThorSync files are assumed to be stored in `RAW_DATA_DIR` in directories with the following structure:
```
raw_data/
+-- {{date_imaged}}
    +-- {{fly_num}}
        +-- {{thorimage_name}}
        +-- {{thorsync_name}}
```

We save the preprocessed outputs to:
```
processed_data/
+-- {{date_imaged}}
    +-- {{fly_num}}
        +-- {{thorimage_name}} # proc_mov_dir
            +-- timestamps.npy
```

In [6]:
date_imaged = '2022-02-11'
fly_num = 1
thorimage_name = 'kiwi_ea_eb_only'
# thorsync_name = 'SyncData001'

# Set these manually if desired
thorimage_folder = RAW_DATA_DIR.joinpath(date_imaged,
                                         str(fly_num),
                                         thorimage_name)

# thorsync_folder = RAW_DATA_DIR.joinpath(date_imaged,
#                                         str(fly_num),
#                                         thorsync_name)

proc_mov_dir = NAS_PROC_DIR.joinpath(date_imaged, str(fly_num), thorimage_name)
proc_mov_dir

PosixPath('/local/matrix/Remy-Data/projects/natural_mixtures/processed_data/2022-02-11/1/kiwi_ea_eb_only')

In [7]:
timestamps = np.load(proc_mov_dir.joinpath('timestamps.npy'), allow_pickle=True).item()
n_timestamps = timestamps['stack_times'].shape[0]
n_timestamps

5070

In [8]:
# downsample timestamps
tsub = 3

n_timestamps_downsampled = timestamps['stack_times'].shape[0]//3

timestamps_ds = copy.deepcopy(timestamps)

stack_times_ds = timestamps['stack_times'][::tsub]
stack_times_ds.shape

(1690,)

In [11]:
timestamps_ds['stack_times'] = stack_times_ds
for k, v in timestamps_ds.items():
    print(f"{k}: {v.shape}")

frame_times: (101400,)
stack_times: (1690,)
scope_ict: (1,)
scope_fct: (1,)
olf_ict: (57,)
olf_fct: (57,)


In [9]:
ds_mov_dir = proc_mov_dir.joinpath(f'downsampled_{tsub}')
display(ds_mov_dir.is_dir())
ds_mov_dir


True

PosixPath('/local/matrix/Remy-Data/projects/natural_mixtures/processed_data/2022-02-11/1/kiwi_ea_eb_only/downsampled_3')

In [12]:
np.save(ds_mov_dir.joinpath('timestamps.npy'), timestamps_ds)